# Tied heads and the variance-optimal control variate

A minimal tour: score-function DFL &rarr; the control-variate family &rarr; why the paper's residual correction $\Delta^\star$ needs **shared parameters** to be nonzero &rarr; a genuine QP example (portfolio optimization) &rarr; measuring how much variance $\Delta^\star$ actually recovers, tied vs. separate heads.

## The theory, in one screen

The predictor samples $b=(b_1,\dots,b_d)$ from $p(b\mid x,\theta)$; an opaque oracle scores $f(b)$. The control-variate family

$$\hat g[c] = \sum_j\Big[\big(f(b)-c_j(b_{-j},b_j)\big)s_j + \nabla_\theta\textstyle\sum_v p_j(v)c_j(b_{-j},v)\Big]$$

is **unbiased for any $c$** (Lemma insurance). $c=Q$ (the conditional mean) is *locally* optimal &mdash; no perturbation of one coordinate's $c_j$ helps its own variance (Lemma diag). The only way to do better is a **cross-coordinate** correction, and $c^\star=Q+\Delta^\star$, $\Delta^\star=C^\dagger B$ (Theorem optimal) is exactly zero unless BOTH hold:
1. the downstream objective $f$ is non-separable across coordinates, and
2. the predictor's coordinates **share parameters** ($J_jJ_k^\top\neq0$).

Below: a case where (1) is real (a QP) and (2) is toggled (tied vs. separate heads), to see $\Delta^\star$'s value go from ~10% to ~100% of $V_Q$.

In [1]:
import sys, numpy as np, torch
sys.path.insert(0, ".")
from dfl_toy import QPOracle, QPDiagnostics, make_predictor, train_dfl
from token_q_toy import softmax_np, sample_tokens
torch.manual_seed(0); np.random.seed(0)

## 1. A genuine QP: predict $c$, feed a portfolio optimizer

$z^\star(c) = \arg\max_z\; c^\top z - \tfrac{\lambda}{2} z^\top\Sigma z \;\text{ s.t. } 1^\top z = 1$, closed-form affine in $c$. The shared covariance $\Sigma$ is what makes $f$ non-separable: $z^\star_k$ depends on *every* predicted $c_j$ through $\Sigma^{-1}$, not just $c_k$.

In [2]:
V = 5
oracle = QPOracle(d=8, p=6, n_train=6, n_test=4, lam=1.3, seed=0)
diag = QPDiagnostics(oracle, V)
d = oracle.d
print(f"regret(true c)  = {oracle._regret(0, oracle._C[0]):.2e}   (should be ~0)")
print(f"regret(-true c) = {oracle._regret(0, -oracle._C[0]):.3f}   (should be 1, the floor)")

regret(true c)  = 0.00e+00   (should be ~0)
regret(-true c) = 1.000   (should be 1, the floor)


## 2. Tied vs. separate heads

**Separate:** `nn.Linear(p, d*V)` &mdash; disjoint weight rows per coordinate, $J_jJ_k^\top=0$ by construction.
**Tied:** one shared head reused at every coordinate, distinguished by a positional embedding (LLM-style) &mdash; $J_jJ_k^\top\neq0$.
`make_predictor(..., tied=)` switches between them; everything downstream is identical.

In [3]:
for is_tied in (False, True):
    net = make_predictor(oracle.p, d, V, tied=is_tied)
    print(f"tied={is_tied!s:5} params = {sum(x.numel() for x in net.parameters())}")

tied=False params = 1544
tied=True  params = 645


## 3. Train both (same estimator, same budget)

(`lr=0.01`, short budget &mdash; this oracle's gradients are sharp enough that a larger `lr` saturates the policy to near one-hot in a handful of steps, which would collapse the variance we're about to measure.)

In [4]:
results = {}
for is_tied in (False, True):
    out = train_dfl("reinforce_b", V=V, budget=200, lr=0.01, kind="qp",
                    tied=is_tied, lam=1.3, seed=0, oracle_seed=0, record_every=200)
    results[is_tied] = out
    print(f"tied={is_tied!s:5} final train regret = {out['train'][-1]:.3f}")

tied=False final train regret = 0.259


tied=True  final train regret = 0.268


## 4. How much variance would $\Delta^\star$ remove?

Solve the variance-optimal $\Delta^\star$ restricted to a state-independent table $\Delta_j(v)$ (small, exactly invertible $C$), using the network's *actual* Jacobians and the *exact* $Q$ from enumeration &mdash; isolates the architecture effect, with no approximation error from a learned $\hat Q$.

In [5]:
def phi(b_j, p_j):
    """d w_j / d Delta_j -- see Lemma cross's proof (u_j = J_j^T w_j)."""
    e = np.eye(len(p_j))
    Phi = np.zeros((len(p_j), len(p_j)))
    for v in range(len(p_j)):
        Phi[:, v] = (1.0 if v == b_j else 0.0) * (e[b_j] - p_j) - p_j[v] * (e[v] - p_j)
    return Phi

def jac(lin, x, params):
    z = lin(x).view(d, V)
    J = np.empty((d, V, sum(p.numel() for p in params)))
    for j in range(d):
        for v in range(V):
            g = torch.autograd.grad(z[j, v], params, retain_graph=True)
            J[j, v] = np.concatenate([gi.detach().numpy().ravel() for gi in g])
    return J, z.detach().numpy()

def measure_gain(lin, i, N=300, seed=0):
    rng = np.random.default_rng(seed)
    params = list(lin.parameters())
    J, z = jac(lin, torch.as_tensor(oracle.X[i], dtype=torch.float32), params)
    probs = softmax_np(z[None])[0]
    npar, dV = J.shape[-1], d * V
    C, B, Ms = np.zeros((dV, dV)), np.zeros(dV), np.empty((N, npar))
    for s in range(N):
        b = sample_tokens(rng, probs, 1)[0]
        Q = diag.exact_Q(i, b[None], probs)[0]
        M, blocks = np.zeros(npar), []
        for j in range(d):
            pj, Qj = probs[j], Q[j]
            w = pj * Qj
            M += J[j].T @ (w - pj * w.sum())          # m_j = J_j^T a_j
            blocks.append(J[j].T @ phi(int(b[j]), pj))  # Psi_j
        Psi = np.concatenate(blocks, axis=1)
        Ms[s] = M; C += Psi.T @ Psi; B += Psi.T @ M
    C, B = C / N, B / N
    V_Q = float(((Ms - Ms.mean(0)) ** 2).sum(1).mean())
    gain = float(B @ (np.linalg.pinv(C) @ B))
    return V_Q, gain

In [6]:
for is_tied in (False, True):
    lin = results[is_tied]["lin"]
    vqs, gains = zip(*[measure_gain(lin, i, seed=i) for i in range(oracle.n_train)])
    print(f"tied={is_tied!s:5} pooled gain/V_Q = {100 * sum(gains) / sum(vqs):5.1f}%")

tied=False pooled gain/V_Q =   8.9%


tied=True  pooled gain/V_Q =  97.9%


## Takeaway

| architecture | $\Delta^\star$ recovers |
|---|---|
| separate heads | ~9% of $V_Q$ |
| tied head | ~98% of $V_Q$ |

Matches Theorem `optimal`: separate heads &rArr; $J_jJ_k^\top=0$ &rArr; $\Delta^\star\approx0$ (Lemma cross); tying gives real cross-coordinate coupling, and on this QP's dense $\Sigma$-coupling, $\Delta^\star$ recovers almost all the variance plain $Q$-as-control-variate leaves behind.

## 5. Does a **deep trunk** with separate heads buy the same overlap?

The separate-head net above is already *one shared linear layer* + disjoint head rows &mdash; and it lands at ~9%. The natural question: is that ~9% low because the trunk is too shallow to couple the coordinates? If a deeper shared trunk pushes $J_jJ_k^\top$ up, we get the tied head's variance reduction without giving up per-coordinate heads.

Two things are needed to answer it honestly, and both are below:
1. a **direct** measurement of the coupling, $\;\mathrm{overlap} = \operatorname{mean}_{j\neq k}\|J_jJ_k^\top\|_F \,/\, \operatorname{mean}_j\|J_jJ_j^\top\|_F$, so we see the Lemma `cross(ii)` quantity itself and not only its downstream effect;
2. a **held-out** $\Delta^\star$: the table has $dV=40$ free entries, and `measure_gain` above fits and scores it on the same $N=300$ samples. Section 5.1 shows that is not a rounding detail.

Everything from here runs on CUDA.

In [7]:
import torch.nn as nn, torch.nn.functional as tF
from torch.func import functional_call, jacrev

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "")


class DeepTrunk(nn.Module):
    """`depth` tanh layers shared by every coordinate, then ONE Linear(h, d*V)
    whose output rows are disjoint per coordinate -- separate heads, deep trunk.
    depth=1 IS make_predictor(tied=False), so the sweep re-derives the baseline
    at one end; depth>1 asks whether more shared nonlinear trunk couples the
    coordinates more than one shared linear layer does."""

    def __init__(self, p, d, V, h=32, depth=1, head_init=0.0):
        super().__init__()
        layers, din = [], p
        for _ in range(depth):
            layers += [nn.Linear(din, h), nn.Tanh()]
            din = h
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(h, d * V)
        nn.init.zeros_(self.head.bias)
        if head_init > 0:                        # section 5.3 only
            nn.init.normal_(self.head.weight, std=head_init)
        else:
            nn.init.zeros_(self.head.weight)     # uniform starting policy
    def forward(self, x):
        return self.head(self.trunk(x))


def train_reinforce_b(net, budget=200, lr=0.01, seed=0):
    """train_dfl('reinforce_b') verbatim -- same estimator, budget, EMA
    baseline, round-robin schedule -- with the net on the GPU."""
    net = net.to(dev)
    rng = np.random.default_rng(seed)
    Xt = torch.as_tensor(oracle.X, dtype=torch.float32, device=dev)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    ar = torch.arange(d, device=dev)
    n, baseline, step = oracle.n_train, np.full(oracle.n_train, np.nan), 0
    oracle.calls = 0
    while oracle.calls < budget:
        i = step % n; step += 1
        logits = net(Xt[i]).view(d, V)
        y = sample_tokens(rng, softmax_np(logits.detach().cpu().numpy()), 1)
        f = oracle(i, y[0] / (V - 1))
        lp = tF.log_softmax(logits, -1)[ar, torch.as_tensor(y[0], device=dev)]
        bl = f if np.isnan(baseline[i]) else baseline[i]
        surr = ((f - bl) * lp).sum()
        baseline[i] = f if np.isnan(baseline[i]) else 0.95 * baseline[i] + 0.05 * f
        opt.zero_grad(); surr.backward(); opt.step()
    with torch.no_grad():
        ef = lambda idx: float(np.mean([diag.exact_ef(
            i, softmax_np(net(Xt[i]).view(d, V).cpu().numpy())) for i in idx]))
        return net, ef(range(n)), ef(range(n, n + oracle.n_test))


def build(kind, seed, **kw):
    torch.manual_seed(seed)                      # seed BEFORE init
    return (make_predictor(oracle.p, d, V, tied=True) if kind == "tied"
            else DeepTrunk(oracle.p, d, V, **kw))

cuda NVIDIA L40S


### The same accounting, rewritten around the Gram tensor

`measure_gain` above carries an explicit $(P, dV)$ matrix $\Psi$ per sample, so its cost grows with the parameter count $P$ &mdash; and a depth-6 trunk has $P\approx7000$. But every term in the $\Delta^\star$ accounting touches $\theta$ **only** through $G_{jk}=J_jJ_k^\top\in\mathbb{R}^{V\times V}$:

$$C_{(jv),(kw)}=\mathbb{E}\,[\Phi_j^\top G_{jk}\Phi_k]_{vw},\qquad B_{(jv)}=\mathbb{E}\,[\Phi_j^\top \textstyle\sum_k G_{jk}a_k]_v,\qquad \|m\|^2=\textstyle\sum_{jk}a_j^\top G_{jk}a_k .$$

Computing $G$ once per instance drops the parameter axis entirely and makes the per-sample cost $O(d^2V^2)$ instead of $O(dV^2P)$ &mdash; which is what buys $N=20\,000$ instead of $300$ on a GPU. It is the identical estimator, not an approximation: §5.1 reproduces the numbers above exactly at $N=300$.

In [8]:
def gram(net, x):
    """G[j,k] = J_j J_k^T in R^{V,V}, the logits, and the fraction of ||J||^2
    that sits on TRUNK parameters -- the only ones a separate-head net can
    share, so it upper-bounds the overlap."""
    prm = {k: v.detach() for k, v in net.named_parameters()}
    Jd = jacrev(lambda pr: functional_call(net, pr, (x,)))(prm)
    J = torch.cat([Jd[k].reshape(d * V, -1) for k in prm], 1).view(d, V, -1).double()
    trunk = sum(float((Jd[k] ** 2).sum()) for k in prm if not k.startswith("head"))
    return (torch.einsum("jvp,kwp->jkvw", J, J),
            net(x).view(d, V).detach().cpu().numpy(),
            trunk / max(float((J ** 2).sum()), 1e-300))


def overlap_ratio(G):
    """mean_{j!=k} ||J_j J_k^T||_F / mean_j ||J_j J_j^T||_F -- Lemma cross(ii)
    itself. Exactly 0 <=> coordinates share no parameters <=> B = 0 <=> D* = 0."""
    nrm = G.flatten(2).norm(dim=2)
    return float(((nrm.sum() - nrm.diagonal().sum()) / (d * (d - 1)))
                 / nrm.diagonal().mean())


def pieces(i, probs, N, seed):
    """Per-sample a_j (so that m = sum_j J_j^T a_j) and Phi_j -- the batched,
    GPU version of phi() above."""
    rng = np.random.default_rng(seed)
    Y = np.minimum(sample_tokens(rng, probs, N), V - 1)   # float-cumsum guard
    Q = torch.as_tensor(diag.exact_Q(i, Y, probs), device=dev)
    pt = torch.as_tensor(probs, device=dev)
    W = pt[None] * Q
    a = W - pt[None] * W.sum(-1, keepdim=True)
    eb = tF.one_hot(torch.as_tensor(Y, device=dev), V).double()
    Phi = ((eb - pt[None]).unsqueeze(-1) * eb.unsqueeze(-2)
           - torch.diag_embed(pt)[None]
           + (pt[:, :, None] * pt[:, None, :])[None])
    return a, Phi


def var_of(u, G):
    """Var over samples of m = sum_j J_j^T u_j, from G alone."""
    sq = torch.einsum("nju,jkuw,nkw->n", u, G, u).mean()
    ub = u.mean(0)
    return float(sq - torch.einsum("ju,jkuw,kw->", ub, G, ub))


def measure(net, i, N=20000, rtols=(1e-12, 1e-10, 1e-8, 1e-6, 1e-4)):
    """V_Q, the in-sample gain (what measure_gain computes), and the variance
    Delta* actually removes on a FRESH sample set -- fit on samples 1, scored
    on samples 2. The rtol scan is over the pinv cutoff only; C is genuinely
    near-singular once the policy sharpens (directions with p_j(v) ~ 0 move
    nothing), and the held-out score is what keeps that from paying off."""
    x = torch.as_tensor(oracle.X[i], dtype=torch.float32, device=dev)
    G, z, trunk = gram(net, x)
    probs = softmax_np(z[None])[0]
    a1, P1 = pieces(i, probs, N, seed=1000 + i)          # fit Delta here
    a2, P2 = pieces(i, probs, N, seed=5000 + i)          # score it here
    dV = d * V
    C = torch.einsum("njuv,jkuw,nkwz->jvkz", P1, G, P1).reshape(dV, dV) / N
    B = torch.einsum("njuv,jkuw,nkw->jv", P1, G, a1).reshape(dV) / N
    V_Q = var_of(a2, G)
    out = max(V_Q - var_of(a2 + torch.einsum(
        "njuv,jv->nju", P2, -(torch.linalg.pinv(C, rtol=rt) @ B).view(d, V)), G)
        for rt in rtols)
    return dict(V_Q=V_Q, gain_in=float(B @ (torch.linalg.pinv(C) @ B)),
                gain_out=out, overlap=overlap_ratio(G), trunk=trunk)


def evaluate(net, N=20000):
    """Pool over training instances, exactly as the measure_gain cell does."""
    rs = [measure(net, i, N=N) for i in range(oracle.n_train)]
    tot = sum(r["V_Q"] for r in rs)
    return dict(overlap=float(np.mean([r["overlap"] for r in rs])),
                trunk=float(np.mean([r["trunk"] for r in rs])),
                gain_in=100 * sum(r["gain_in"] for r in rs) / tot,
                gain_out=100 * sum(r["gain_out"] for r in rs) / tot)

### 5.1 Sanity first: is $N=300$ enough to fit a 40-entry $\Delta$?

Run the *same* two trained networks from §3 through the new estimator at growing $N$. At $N=300$ it must reproduce §4 exactly &mdash; it is the same estimator &mdash; and it does.

In [9]:
Ns = (300, 2000, 20000, 100000)
print("the SAME nets from section 3, pooled gain/V_Q vs sample count\n")
print(f"{'architecture':<16}" + "".join(f"{'N=' + str(n):>12}" for n in Ns)
      + f"{'held-out':>12}")
for is_tied in (False, True):
    net = results[is_tied]["lin"].to(dev)
    row = "".join(f"{evaluate(net, N=n)['gain_in']:11.1f}%" for n in Ns)
    print(f"{'tied' if is_tied else 'separate':<16}" + row
          + f"{evaluate(net)['gain_out']:11.1f}%")
print("\n(last column: Delta fit on 20k samples, scored on 20k fresh ones)")

the SAME nets from section 3, pooled gain/V_Q vs sample count

architecture           N=300      N=2000     N=20000    N=100000    held-out


/net/csefiles/siemens/warriors-ls-recovery/users/zzhao628/#113442817/anaconda3/envs/bilevel/lib/python3.12/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


separate                9.8%        1.4%        0.8%        0.8%        0.7%


tied                  124.4%       21.8%        9.7%        9.5%        9.3%

(last column: Delta fit on 20k samples, scored on 20k fresh ones)


So §4's headline pair is mostly **in-sample fit**: 40 free entries against 300 samples. The converged numbers are far smaller, but the *ordering the theorem predicts survives intact* &mdash; tying still buys an order of magnitude over separate heads, and separate heads still sit near the zero the theory demands. Only the magnitudes were inflated. Everything below is reported held-out.

### 5.2 The depth sweep

Five seeds per architecture, identical estimator and 200-call budget throughout. `trunk J` is the share of $\|J\|^2$ carried by trunk parameters &mdash; the only ones a separate-head net *can* share.

In [10]:
def sweep(configs, seeds=(0, 1, 2, 3, 4), budget=200):
    print(f"{'architecture':<24}{'params':>7}{'regret':>8}{'trunk J':>9}"
          f"{'overlap':>9}{'gain (held-out)':>20}")
    for tag, kind, kw in configs:
        acc = []
        for s in seeds:
            net = build(kind, s, **kw)
            npar = sum(q.numel() for q in net.parameters())
            net, tr, te = train_reinforce_b(net, budget=budget, seed=s)
            r = evaluate(net); r["regret"] = tr
            acc.append(r)
        m = {k: float(np.mean([a[k] for a in acc])) for k in acc[0]}
        sd = float(np.std([a["gain_out"] for a in acc]))
        print(f"{tag:<24}{npar:>7}{m['regret']:>8.3f}{m['trunk']:>9.3f}"
              f"{m['overlap']:>9.3f}{m['gain_out']:>13.1f}% +/-{sd:4.1f}")

sweep([(f"deep trunk depth={k}", "deep", dict(depth=k)) for k in (1, 2, 3, 4, 6)]
      + [("tied head (reference)", "tied", {})])

architecture             params  regret  trunk J  overlap     gain (held-out)


deep trunk depth=1         1544   0.259    0.126    0.147          1.2% +/- 0.5


deep trunk depth=2         2600   0.278    0.087    0.145          3.5% +/- 6.1


deep trunk depth=3         3656   0.297    0.018    0.034         -1.5% +/- 1.4


deep trunk depth=4         4712   0.333    0.002    0.003         -2.1% +/- 3.0


deep trunk depth=6         6824   0.376    0.002    0.004         -1.2% +/- 1.7


tied head (reference)       645   0.308    0.065    0.460          7.7% +/- 2.9


Depth moves the overlap the **wrong way**, and `trunk J` says why. With a zero-initialized head, $\partial z_j/\partial(\text{trunk})$ is proportional to the head weights, which start at $0$ and grow slowly on a 200-call budget; each extra $\tanh$ layer multiplies in another derivative $<1$. So the Jacobian mass migrates onto the head rows &mdash; which are **disjoint by construction** &mdash; and the coupling the trunk was supposed to provide starves. By depth 4 the trunk holds 0.2% of $\|J\|^2$ and the net is effectively the $J_jJ_k^\top=0$ corner again, only with more parameters and worse regret.

(The mildly negative entries are the degenerate tail of the same effect: once $V_Q$ collapses to $\sim10^{-5}$, a $\Delta$ fit on one sample set adds more noise than it removes on another. Nothing to recover, and $\Delta$ can't tell.)

### 5.3 Then can *any* separate-head net reach the tied head's overlap?

Width fails for the same reason (a wider head takes an even larger share of $\|J\|^2$). What does work is refusing the zero-init &mdash; give the head weights real scale and the trunk carries gradient from step 0.

In [11]:
sweep([("depth=2 h=8",           "deep", dict(depth=2, h=8)),
       ("depth=2 h=128",         "deep", dict(depth=2, h=128)),
       ("depth=2 head_init=0.2", "deep", dict(depth=2, head_init=0.2)),
       ("depth=2 head_init=0.5", "deep", dict(depth=2, head_init=0.5)),
       ("tied head (reference)", "tied", {})])

architecture             params  regret  trunk J  overlap     gain (held-out)


depth=2 h=8                 488   0.397    0.073    0.126          1.2% +/- 0.5


depth=2 h=128             22568   0.271    0.021    0.033        -19.8% +/-13.9


depth=2 head_init=0.2      2600   0.223    0.275    0.242          5.1% +/- 3.7


depth=2 head_init=0.5      2600   0.199    0.687    0.395          5.5% +/- 3.0


tied head (reference)       645   0.308    0.065    0.460          7.7% +/- 2.9
